# Datamart checks

## 1. Start Spark and load the files

In [1]:
import pyspark
import pyspark.sql.functions as F
from pyspark.sql.functions import col
import os
import pandas as pd

spark = pyspark.sql.SparkSession.builder.appName("dev").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

DATA_DIR = "../data"
DATAMART = "../datamart"

SOURCES = {
    "attributes": "features_attributes.csv",
    "financials": "features_financials.csv",
    "clickstream": "feature_clickstream.csv",
    "loan_daily": "lms_loan_daily.csv",
}

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/28 11:20:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/28 11:20:29 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## 2. Bronze checks

In [2]:
for name in SOURCES:
    files = sorted(os.listdir(f"{DATAMART}/bronze/{name}"))
    print(f"{name}: {len(files)} partitions, {files[0]} to {files[-1]}")

attributes: 25 partitions, bronze_attributes_2023_01_01.csv to bronze_attributes_2025_01_01.csv
financials: 25 partitions, bronze_financials_2023_01_01.csv to bronze_financials_2025_01_01.csv
clickstream: 24 partitions, bronze_clickstream_2023_01_01.csv to bronze_clickstream_2024_12_01.csv
loan_daily: 35 partitions, bronze_loan_daily_2023_01_01.csv to bronze_loan_daily_2025_11_01.csv


In [3]:
for name, file_name in SOURCES.items():
    raw = spark.read.csv(f"{DATA_DIR}/{file_name}", header=True, escape='"')
    bronze = spark.read.csv(f"{DATAMART}/bronze/{name}/*", header=True, escape='"')
    print(f"{name}: raw {raw.count()} rows, bronze {bronze.count()} rows, "
          f"raw rows not in bronze {raw.exceptAll(bronze).count()}, "
          f"bronze rows not in raw {bronze.exceptAll(raw).count()}")

attributes: raw 12500 rows, bronze 12500 rows, raw rows not in bronze 0, bronze rows not in raw 0
financials: raw 12500 rows, bronze 12500 rows, raw rows not in bronze 0, bronze rows not in raw 0
clickstream: raw 215376 rows, bronze 215376 rows, raw rows not in bronze 0, bronze rows not in raw 0
loan_daily: raw 137500 rows, bronze 137500 rows, raw rows not in bronze 0, bronze rows not in raw 0


**Findings:**
- Each source is stored as one partition per month
- Every raw row is in bronze and bronze has no other rows (0 differences in both directions), so bronze matches the raw data

## 3. Silver checks

In [4]:
silver = {name: spark.read.parquet(f"{DATAMART}/silver/{name}/*") for name in SOURCES}
for name, df in silver.items():
    print(f"{name}: {df.count()} rows, {len(df.columns)} columns")
silver["financials"].printSchema()

attributes: 12500 rows, 6 columns
financials: 12500 rows, 23 columns
clickstream: 215376 rows, 22 columns
loan_daily: 137500 rows, 15 columns
root
 |-- Customer_ID: string (nullable = true)
 |-- Annual_Income: float (nullable = true)
 |-- Monthly_Inhand_Salary: float (nullable = true)
 |-- Num_Bank_Accounts: integer (nullable = true)
 |-- Num_Credit_Card: integer (nullable = true)
 |-- Interest_Rate: integer (nullable = true)
 |-- Num_of_Loan: integer (nullable = true)
 |-- Type_of_Loan: string (nullable = true)
 |-- Delay_from_due_date: integer (nullable = true)
 |-- Num_of_Delayed_Payment: integer (nullable = true)
 |-- Changed_Credit_Limit: float (nullable = true)
 |-- Num_Credit_Inquiries: integer (nullable = true)
 |-- Credit_Mix: string (nullable = true)
 |-- Outstanding_Debt: float (nullable = true)
 |-- Credit_Utilization_Ratio: float (nullable = true)
 |-- Credit_History_Age: string (nullable = true)
 |-- Payment_of_Min_Amount: string (nullable = true)
 |-- Total_EMI_per_month

In [5]:
def missing_counts(df):
    return df.select([F.sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).toPandas().T[0]

for name in ["attributes", "financials"]:
    bronze = spark.read.csv(f"{DATAMART}/bronze/{name}/*", header=True, escape='"')
    comparison = pd.DataFrame({"missing in bronze": missing_counts(bronze),
                               "missing in silver": missing_counts(silver[name])})
    display(comparison[comparison["missing in silver"] > 0])

,missing in bronze,missing in silver
Age,0,319
SSN,0,703
Occupation,0,880


,missing in bronze,missing in silver
Amount_invested_monthly,0.0,558
Annual_Income,0.0,114
Changed_Credit_Limit,0.0,254
Credit_Mix,0.0,2611
Interest_Rate,0.0,270
Monthly_Balance,0.0,1
Num_Bank_Accounts,0.0,171
Num_Credit_Card,0.0,296
Num_Credit_Inquiries,0.0,195
Num_of_Delayed_Payment,0.0,189


In [6]:
silver["attributes"].select("Age").summary("min", "1%", "50%", "99%", "max").show()
silver["financials"].select("Annual_Income", "Num_Bank_Accounts", "Num_Credit_Card", "Interest_Rate", "Num_of_Loan", "Num_of_Delayed_Payment", 
                        "Num_Credit_Inquiries", "Total_EMI_per_month", "Amount_invested_monthly", "Monthly_Balance", "Credit_History_Months").summary("min", "50%", "max").toPandas().set_index("summary").T

+-------+---+
|summary|Age|
+-------+---+
|    min| 14|
|     1%| 15|
|    50%| 33|
|    99%| 55|
|    max| 56|
+-------+---+



summary,min,50%,max
Annual_Income,7005.93,37006.6,179987.28
Num_Bank_Accounts,0,5,11
Num_Credit_Card,0,5,11
Interest_Rate,1,13,34
Num_of_Loan,0,3,9
Num_of_Delayed_Payment,0,14,28
Num_Credit_Inquiries,0,6,17
Total_EMI_per_month,0.0,69.24668,7539.0
Amount_invested_monthly,0.0,127.33208,1977.326
Monthly_Balance,0.382558,339.74963,1463.7924


In [7]:
silver["loan_daily"].groupBy("mob").agg(F.round(F.avg((col("dpd") >= 30).cast("int")), 3).alias("bad_rate")).orderBy("mob").show()

+---+--------+
|mob|bad_rate|
+---+--------+
|  0|     0.0|
|  1|   0.051|
|  2|   0.101|
|  3|    0.17|
|  4|   0.227|
|  5|   0.283|
|  6|   0.288|
|  7|   0.288|
|  8|   0.288|
|  9|   0.288|
| 10|   0.288|
+---+--------+



**Findings:**
- No rows dropped: all silver tables have same row count as bronze
- Values made missing match the EDA counts
- Every cleaned column now lies in its plausible range per EDA
- The bad rate by mob is the same as the EDA (28.8% from mob 6)

## 4. Gold checks

In [8]:
labels = spark.read.parquet(f"{DATAMART}/gold/label_store/*")
features = spark.read.parquet(f"{DATAMART}/gold/feature_store/*")

print("label store:", labels.count(), "rows,", labels.select("loan_id").distinct().count(), "loans")
print("feature store:", features.count(), "rows,", features.select("Customer_ID", "snapshot_date").distinct().count(), "applications,", len(features.columns), "columns")

labels.groupBy("label_def").agg(F.count("*").alias("loans"), F.round(F.avg("label"), 3).alias("bad_rate"),
                            F.min("snapshot_date").alias("first_label_date"),
                            F.max("snapshot_date").alias("last_label_date")).show()

label store: 12500 rows, 12500 loans
feature store: 12500 rows, 12500 applications, 68 columns
+----------+-----+--------+----------------+---------------+
| label_def|loans|bad_rate|first_label_date|last_label_date|
+----------+-----+--------+----------------+---------------+
|30dpd_6mob|12500|   0.288|      2023-07-01|     2025-07-01|
+----------+-----+--------+----------------+---------------+



In [9]:
applications = features.withColumnRenamed("snapshot_date", "loan_start_date")
matched = labels.join(applications, ["Customer_ID", "loan_start_date"], "inner")
print("labels with features:", matched.count())
print("labels without features:", labels.join(applications, ["Customer_ID", "loan_start_date"], "left_anti").count())

labels with features: 12500
labels without features: 0


In [10]:
missing = missing_counts(features)
missing[missing > 0]

Age                         319
Annual_Income               114
Num_Bank_Accounts           171
Num_Credit_Card             296
Interest_Rate               270
Num_of_Loan                 567
Num_of_Delayed_Payment      189
Changed_Credit_Limit        254
Num_Credit_Inquiries        195
Total_EMI_per_month         397
Amount_invested_monthly     558
Monthly_Balance               1
EMI_to_Monthly_Salary       397
fe_1_avg                   3526
fe_2_avg                   3526
fe_3_avg                   3526
fe_4_avg                   3526
fe_5_avg                   3526
fe_6_avg                   3526
fe_7_avg                   3526
fe_8_avg                   3526
fe_9_avg                   3526
fe_10_avg                  3526
fe_11_avg                  3526
fe_12_avg                  3526
fe_13_avg                  3526
fe_14_avg                  3526
fe_15_avg                  3526
fe_16_avg                  3526
fe_17_avg                  3526
fe_18_avg                  3526
fe_19_av

In [11]:
loan_columns = ["loan_id", "loan_amt", "due_amt", "paid_amt", "overdue_amt", "balance", "installments_missed", "first_missed_date", "dpd", "mob", "label"]
print("loan columns in the feature store:", [c for c in features.columns if c in loan_columns])

loan columns in the feature store: []


In [12]:
application_months = silver["attributes"].select("Customer_ID", col("snapshot_date").alias("application_date"))
history = silver["clickstream"].join(application_months, "Customer_ID")
up_to_application = history.filter(col("snapshot_date") <= col("application_date")).groupBy("Customer_ID").agg(F.avg("fe_1").alias("expected_fe_1_avg"))
all_months = history.groupBy("Customer_ID").agg(F.avg("fe_1").alias("all_months_fe_1_avg"))
check = features.select("Customer_ID", "fe_1_avg").join(up_to_application, "Customer_ID").join(all_months, "Customer_ID")

check.select(
    F.count("*").alias("customers"),
    F.max(F.abs(col("fe_1_avg") - col("expected_fe_1_avg"))).alias("max_diff_up_to_application"),
    F.sum((F.abs(col("fe_1_avg") - col("all_months_fe_1_avg")) > 1e-6).cast("int")).alias("differs_from_all_months"),
).show()
check.filter(col("Customer_ID") == "CUS_0x10ee").show()

+---------+--------------------------+-----------------------+
|customers|max_diff_up_to_application|differs_from_all_months|
+---------+--------------------------+-----------------------+
|     8974|                       0.0|                   8970|
+---------+--------------------------+-----------------------+

+-----------+------------------+------------------+-------------------+
|Customer_ID|          fe_1_avg| expected_fe_1_avg|all_months_fe_1_avg|
+-----------+------------------+------------------+-------------------+
| CUS_0x10ee|122.83333333333333|122.83333333333333|  68.95833333333333|
+-----------+------------------+------------------+-------------------+



**Findings:**
- No rows are duplicated or dropped for both stores
- Every label joins to exactly one feature row on customer and application month
- Missing values are kept as-is for the model pipeline - this includes applicants with no clickstream history
- No loan repayment column is in the feature store
- The clickstream averages match an independent recalculation over months up to the application, and differ from the all-months average

## 5. Test attribute cleaning rules on dummy values

In [13]:
import sys
sys.path.append("..")
import utils.data_processing_silver_table

test_dir = "/tmp/silver_rule_test/"
os.makedirs(test_dir, exist_ok=True)
pd.DataFrame({
    "Customer_ID": ["T1", "T2", "T3", "T4", "T5"],
    "Name": ['ODonnell"d', "Ann", "Bob", "Cy", "Di"],
    "Age": ["23_", "-500", "8678", "14", "45"],
    "SSN": ["123-45-6789", "#F%$D@*&8", "123-45-6789", "123-45-6789", "123-45-6789"],
    "Occupation": ["Doctor", "_______", "Lawyer", "Writer", "Teacher"],
    "snapshot_date": ["1900-01-01"] * 5,
}).to_csv(test_dir + "bronze_attributes_1900_01_01.csv", index=False)

utils.data_processing_silver_table.process_silver_attributes_table("1900-01-01", test_dir, test_dir, spark).show()

loaded from: /tmp/silver_rule_test/bronze_attributes_1900_01_01.csv row count: 5
saved to: /tmp/silver_rule_test/silver_attributes_1900_01_01.parquet
+-----------+----------+----+-----------+----------+-------------+
|Customer_ID|      Name| Age|        SSN|Occupation|snapshot_date|
+-----------+----------+----+-----------+----------+-------------+
|         T1|ODonnell"d|  23|123-45-6789|    Doctor|   1900-01-01|
|         T2|       Ann|NULL|       NULL|      NULL|   1900-01-01|
|         T3|       Bob|NULL|123-45-6789|    Lawyer|   1900-01-01|
|         T4|        Cy|  14|123-45-6789|    Writer|   1900-01-01|
|         T5|        Di|  45|123-45-6789|   Teacher|   1900-01-01|
+-----------+----------+----+-----------+----------+-------------+



**Findings:**
- Rules behave as expected